# Tingen City Background — Prompt-to-Result Demo

A runnable walkthrough of how the explorable **city background** is made: prompt a top-down city,
relabel it, split it into a **background-only** layer and a **buildings-only** layer, then compose
those in the Godot scene with per-building colliders. Each step is a demo cell — the prompt, and the
`gpt-image-1` call that turns it into the result that feeds the next step.

**Tooling note / attribution.** Steps run through OpenAI's **`gpt-image-1`** Images API
(`generate_tingen_image2.py`). A few edits were instead done by hand in the **ChatGPT in-app image
generator (GPT-Image, the "image 2" tool)** — notably the building-strip that produced the current
`map_bg.png` and assorted `my_assets/` exports. Those aren't in any script/manifest, so they're
credited inline where they occur.

**The one constraint:** `gpt-image-1` has **no seed**. Consistency between steps comes only from
(1) passing the previous result as the **reference image** and (2) **`input_fidelity=high`**, which
holds the layout while you change one thing. (`low` ~$0.02 · `high` ~$0.25 per image — draft low, ship high.)

## Setup

In [ ]:
# pip install requests
import os, base64, requests
from pathlib import Path

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

def gen_image(prompt, ref=None, size='1024x1024', fidelity='high',
              background='opaque', quality='high', out='out.png'):
    """prompt -> result PNG. With a ref -> /images/edits (multipart, honors input_fidelity).
    Without a ref -> /images/generations (pure text-to-image). Returns the output path."""
    h = {'Authorization': f'Bearer {OPENAI_API_KEY}'}
    if ref:
        files = {'image[]': (Path(ref).name, open(ref, 'rb'), 'image/png')}
        data = {'model': 'gpt-image-1', 'prompt': prompt, 'size': size, 'quality': quality,
                'background': background, 'output_format': 'png', 'input_fidelity': fidelity, 'n': 1}
        r = requests.post('https://api.openai.com/v1/images/edits', headers=h,
                          files=files, data=data, timeout=1800)
    else:
        body = {'model': 'gpt-image-1', 'prompt': prompt, 'size': size, 'quality': quality,
                'background': background, 'output_format': 'png', 'n': 1}
        r = requests.post('https://api.openai.com/v1/images/generations',
                          headers={**h, 'Content-Type': 'application/json'}, json=body, timeout=1800)
    r.raise_for_status()
    Path(out).write_bytes(base64.b64decode(r.json()['data'][0]['b64_json']))
    print('wrote', out)
    return out

## Step 1 — Prompt the city (text → image)
The opening prompt. Pure text-to-image (no reference); style + the important features are all in the prose.

In [ ]:
PROMPT_CITY = (
    'Top-down 2D game world map, late Victorian Britain working-class district, hand-painted '
    'cartographic style. Dense brick row houses, small workshops, corner pubs, warehouses, tenements, '
    'narrow alleys, irregular block layouts, varied roof heights and footprints. Dark slate roofs mixed '
    'with weathered red brick roofs, numerous chimneys, soot-stained walls, patched roofs, aged stone '
    'roads. Muted industrial color palette, overcast British daylight, low saturation, dusty browns, '
    'faded reds, gray-blue slate, subtle atmospheric haze. Not wealthy, lived-in and practical. Strong '
    'shape readability for a video game map. High visual variety between buildings while maintaining '
    'coherent urban planning. Detailed but not photorealistic. '
    'Lord of the Mysteries / Tingen-inspired Victorian city atmosphere.'
)

gen_image(PROMPT_CITY, ref=None, size='1024x1024', out='city_raw.png')
# Result: a top-down working-class district. The model auto-letters streets/landmarks, and garbles
# some of it (e.g. GREENMARKET 'SQUARK') -- Step 2 fixes the names.

## Step 2 — Relabel to canon
Edit the image with itself as the reference at `input_fidelity=high`, changing **only** the text.
Because AI image tools garble lettering, the reliable route is to relabel a handful of districts and
hand-fix spelling in Preview/Photopea — but here's the scriptable version.

In [ ]:
PROMPT_RELABEL = (
    'Keep this exact city - same style, same layout, same top-down angle. Change ONLY the text labels '
    "to: Iron Cross Market, The Laughing Eel, Saint Selena's Cathedral, the Docks. Spell every label "
    'exactly as written. Make no other changes.'
)

gen_image(PROMPT_RELABEL, ref='city_raw.png', fidelity='high', out='city_labeled.png')
# Result: same art, canon names.

## Step 3a — Strip the buildings → background only
First of the two layers: remove every building, keep the walkable ground. The empty lots barely need
detail (buildings cover them) — they just need to read as streets.

> The current `my_assets/map_bg.png` was actually produced this way **by hand in the ChatGPT in-app
> image tool** (upload the city, "remove the buildings"). The scriptable equivalent:

In [ ]:
PROMPT_STRIP_BG = (
    'Remove ALL buildings, rooftops and labels. Keep ONLY the cobblestone streets, lanes, squares and '
    'the empty ground between them, in the exact same layout. Flat top-down. Empty paved/dirt lots where '
    'buildings were.'
)

gen_image(PROMPT_STRIP_BG, ref='city_labeled.png', fidelity='high', background='opaque', out='map_bg.png')
# Result: the bare walkable ground (the background layer).

## Step 3b — Buildings only → transparent layer
The mirror edit: keep **only** the buildings, in their exact positions, on a **transparent** background.
Sharing the same reference + layout as 3a means the buildings drop back onto the ground at the same
coordinates. If a building comes back on a white halo instead of clean alpha, run `key_building.py`
(KEY border-white → ERODE the fringe → BLEED nearest colour so linear filtering never samples white).

In [ ]:
PROMPT_BUILDINGS = (
    'Keep ONLY the buildings and rooftops, in their exact same positions and footprints. Remove ALL '
    'streets, ground and labels. Transparent background. Flat top-down, same layout.'
)

gen_image(PROMPT_BUILDINGS, ref='city_labeled.png', fidelity='high',
          background='transparent', out='buildings_layer.png')
# Optional alpha clean-up for a single building compound:
#   python3 key_building.py buildings_layer.png buildings_clean.png --white 222 --erode 2
# Result: the buildings on transparency (the obstacle layer).

## Step 4 — Compose in the Godot scene (background + buildings + colliders)
Assemble the two layers in the `.tscn` and make the buildings **solid** so the player/agents route
around them. The real composed scene is `tingen/scenes/CityBlocks.tscn`: one walkable `Ground` polygon
(the background) + a `Buildings` node of `StaticBody2D`s, each with a `Polygon2D` (visual) **and** a
`CollisionPolygon2D` (the footprint collider):

```gdscript
[node name="Ground" type="Polygon2D" parent="."]                          # the background / walkable floor
polygon = PackedVector2Array(0, 0, 2400, 0, 2400, 1700, 0, 1700)

[node name="B_r0_c0" type="StaticBody2D" parent="Buildings"]               # one building
[node name="Polygon2D" type="Polygon2D" parent="Buildings/B_r0_c0"]        # building visual (from 3b)
polygon = PackedVector2Array(-76.8, -58.8, 76.8, -58.8, 76.8, 58.8, -76.8, 58.8)
[node name="CollisionPolygon2D" type="CollisionPolygon2D" parent="Buildings/B_r0_c0"]   # THE collider
polygon = PackedVector2Array(-76.8, -58.8, 76.8, -58.8, 76.8, 58.8, -76.8, 58.8)
```

- **Background:** `map_bg.png` as the ground (`texture_filter = Linear` for smooth upscale of painted art).
- **Buildings:** the keyed sprite(s) from 3b stamped at their footprint, each in a `StaticBody2D` +
  `CollisionPolygon2D` → solid obstacle.
- Hold the camera at gameplay zoom; the collider footprints are what the player and NPC agents route around.

**TL;DR:** prompt the top-down city → relabel to canon → split into a background-only layer (3a) and a
buildings-only layer (3b) with two aligned `input_fidelity=high` edits → compose in Godot as `map_bg`
ground + building sprites, each wrapped in a `StaticBody2D` + `CollisionPolygon2D`.